# Company Earnings Results 2019–2026: starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/company_earnings_results_sample.csv', recursive=True) or ['company_earnings_results_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df['report_year'] = pd.to_datetime(df.report_date).dt.year
df.pivot_table(index='report_year', columns='period_type', values='result_id', aggfunc='count').fillna(0).astype(int)

In [ ]:
bm = df[df.beat_or_miss.notna()]
bm.groupby('report_year').beat_or_miss.value_counts(normalize=True).unstack().plot(figsize=(9, 3.5), marker='o', title='Beat / miss share (sample)')
bm.groupby('sector').beat_or_miss.value_counts(normalize=True).unstack().dropna().head(15)

In [ ]:
df[df.period == 'FY'].sort_values('revenue_usd', ascending=False).drop_duplicates('company_domain').head(20)[['company_name', 'fiscal_year', 'revenue_usd', 'net_income_usd', 'country']]

## Get the full record from the Radar API

The sample has headline figures per period. The API returns the company's full event timeline, including guidance and every reporting article. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/events', headers={'Authorization': f'Bearer {key}'},
                      json={'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp